# Linear Regression with Gaussian GLM

**Topics:** Linear Regression, Diagnostics, Inference, Model Checking

## Overview

This notebook provides a comprehensive introduction to linear regression using the GLM framework. We'll cover model fitting, diagnostics, inference, and comparison—going beyond the quickstart to show best practices for real-world regression analysis.

## What You'll Learn

- Fit Gaussian GLM (linear regression)
- Interpret coefficients and summary statistics
- Check model assumptions (residual plots)
- Perform hypothesis tests (Wald tests)
- Compute confidence intervals
- Compare nested models
- Evaluate prediction quality

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aurora.models import fit_glm
from aurora.inference import wald_test, confidence_intervals
from aurora.inference.diagnostics import glm_diagnostics
from aurora.validation.metrics import root_mean_squared_error, mean_absolute_error

sns.set_style('whitegrid')
np.random.seed(42)

## 2. Generate Realistic Data

Simulate employee salary data with multiple predictors:
- `experience`: Years of experience
- `education`: Education level (1=HS, 2=Bachelor, 3=Master, 4=PhD)
- `performance`: Performance rating (0-100)
- `salary`: Annual salary in thousands

In [ ]:
n = 200

# Generate predictors
experience = np.random.uniform(0, 20, n)  # 0-20 years
education = np.random.choice([1, 2, 3, 4], n, p=[0.2, 0.4, 0.3, 0.1])
performance = np.random.uniform(50, 100, n)

# True relationship (what we want to recover)
# salary = 40 + 2*experience + 5*education + 0.3*performance + noise
true_salary = (
    40  # Base salary
    + 2 * experience  # +$2k per year experience
    + 5 * education  # +$5k per education level
    + 0.3 * performance  # +$0.3k per performance point
)

# Add realistic noise
noise = np.random.randn(n) * 5  # SD = $5k
salary = true_salary + noise

# Create DataFrame
df = pd.DataFrame({
    'experience': experience,
    'education': education,
    'performance': performance,
    'salary': salary,
    'true_salary': true_salary
})

print(f"Generated data for {n} employees")
print(f"\nSummary statistics:")
print(df[['experience', 'education', 'performance', 'salary']].describe())

# Correlation matrix
print(f"\nCorrelations with salary:")
print(df[['experience', 'education', 'performance', 'salary']].corr()['salary'].sort_values(ascending=False))

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Experience vs Salary
axes[0, 0].scatter(df['experience'], df['salary'], alpha=0.5, edgecolor='k', linewidth=0.5)
axes[0, 0].set_xlabel('Experience (years)')
axes[0, 0].set_ylabel('Salary ($1000s)')
axes[0, 0].set_title('Salary vs Experience')

# Education vs Salary
education_labels = {1: 'HS', 2: 'BS', 3: 'MS', 4: 'PhD'}
df['education_label'] = df['education'].map(education_labels)
sns.boxplot(data=df, x='education_label', y='salary', ax=axes[0, 1], 
            order=['HS', 'BS', 'MS', 'PhD'])
axes[0, 1].set_xlabel('Education Level')
axes[0, 1].set_ylabel('Salary ($1000s)')
axes[0, 1].set_title('Salary by Education')

# Performance vs Salary
axes[1, 0].scatter(df['performance'], df['salary'], alpha=0.5, edgecolor='k', linewidth=0.5)
axes[1, 0].set_xlabel('Performance Rating')
axes[1, 0].set_ylabel('Salary ($1000s)')
axes[1, 0].set_title('Salary vs Performance')

# Salary distribution
axes[1, 1].hist(df['salary'], bins=30, edgecolor='k', alpha=0.7)
axes[1, 1].axvline(df['salary'].mean(), color='r', linestyle='--', label=f'Mean = {df["salary"].mean():.1f}')
axes[1, 1].set_xlabel('Salary ($1000s)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Salary Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Fit the Model

### Full Model with All Predictors

In [ ]:
# Design matrix
X = np.column_stack([
    np.ones(n),  # Intercept
    df['experience'],
    df['education'],
    df['performance']
])
y = df['salary'].values

# Fit Gaussian GLM
# Note: We already included intercept in X, so set fit_intercept=False
result = fit_glm(
    X=X,
    y=y,
    family='gaussian',
    fit_intercept=False  # We manually included the intercept
)

print(f"Model converged: {result.converged_}")
print(f"Iterations: {result.n_iter_}")
print(f"\n" + "="*60)
print(result.summary())
print("="*60)

## 5. Interpret Coefficients

**Coefficient Interpretation** (holding other variables constant):
- **Intercept (β₀)**: Expected salary when all predictors = 0
- **Experience (β₁)**: Expected salary increase per additional year
- **Education (β₂)**: Expected salary increase per education level
- **Performance (β₃)**: Expected salary increase per rating point

**Statistical Significance:**
- p-value < 0.05: Significant at 5% level
- p-value < 0.01: Highly significant
- p-value < 0.001: Very highly significant

In [ ]:
# Extract coefficients
coef_names = ['Intercept', 'Experience', 'Education', 'Performance']
true_coefs = [40, 2, 5, 0.3]

# When fit_intercept=False, all coefficients are in result.coef_
# (including the intercept as the first element)
all_coefs = result.coef_
all_std_errors = result.std_errors_
all_p_values = result.p_values_

coef_df = pd.DataFrame({
    'Predictor': coef_names,
    'True': true_coefs,
    'Estimated': all_coefs,
    'Std Error': all_std_errors,
    'p-value': all_p_values
})
coef_df['Difference'] = coef_df['Estimated'] - coef_df['True']
coef_df['Significant'] = coef_df['p-value'] < 0.05

print("\nCoefficient Recovery:")
print(coef_df.to_string(index=False))

# Plot coefficient comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(coef_names))
width = 0.35

ax.bar(x - width/2, true_coefs, width, label='True', alpha=0.7, color='green')
ax.bar(x + width/2, all_coefs, width, label='Estimated', alpha=0.7, color='blue')
ax.errorbar(x + width/2, all_coefs, yerr=1.96*all_std_errors, 
            fmt='none', color='black', capsize=5, label='95% CI')

ax.set_ylabel('Coefficient Value')
ax.set_title('True vs Estimated Coefficients')
ax.set_xticks(x)
ax.set_xticklabels(coef_names)
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Model Diagnostics

Check model assumptions:
1. **Linearity**: Relationship is linear
2. **Homoscedasticity**: Constant variance of residuals
3. **Normality**: Residuals are normally distributed
4. **Independence**: Observations are independent

In [ ]:
# Compute diagnostics
diagnostics = glm_diagnostics(result)

residuals = diagnostics.response_residuals
fitted = result.predict(X)
standardized_residuals = diagnostics.studentized_residuals

# Diagnostic plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(fitted, residuals, alpha=0.5, edgecolor='k', linewidth=0.5)
axes[0, 0].axhline(0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted\n(Check linearity & homoscedasticity)')

# 2. Q-Q Plot
from scipy import stats
stats.probplot(standardized_residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Normal Q-Q Plot\n(Check normality)')

# 3. Scale-Location Plot
sqrt_std_resid = np.sqrt(np.abs(standardized_residuals))
axes[1, 0].scatter(fitted, sqrt_std_resid, alpha=0.5, edgecolor='k', linewidth=0.5)
axes[1, 0].set_xlabel('Fitted Values')
axes[1, 0].set_ylabel('√|Standardized Residuals|')
axes[1, 0].set_title('Scale-Location Plot\n(Check homoscedasticity)')

# 4. Residuals Histogram
axes[1, 1].hist(standardized_residuals, bins=30, edgecolor='k', alpha=0.7, density=True)
x_range = np.linspace(standardized_residuals.min(), standardized_residuals.max(), 100)
axes[1, 1].plot(x_range, stats.norm.pdf(x_range, 0, 1), 'r-', lw=2, label='N(0,1)')
axes[1, 1].set_xlabel('Standardized Residuals')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Residuals Distribution\n(Check normality)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 7. Hypothesis Testing

### Test 1: Individual Coefficients (Wald Tests)

Already shown in summary. Each p-value tests:
- H₀: βⱼ = 0 (no effect)
- H₁: βⱼ ≠ 0 (has effect)

### Test 2: Multiple Coefficients (Joint Test)

In [ ]:
# Test if education AND performance jointly have no effect
# H0: β_education = 0 AND β_performance = 0
contrast_matrix = np.array([
    [0, 0, 1, 0],  # β_education = 0
    [0, 0, 0, 1]   # β_performance = 0
])

# Since we used fit_intercept=False, the intercept is part of coef_
# So we need to tell wald_test not to look for a separate intercept
test_result = wald_test(
    result,
    contrast=contrast_matrix,
    include_intercept=False
)

print("\nJoint Hypothesis Test:")
print(f"H0: Education = 0 AND Performance = 0")
print(f"Test statistic: {test_result['statistic']:.4f}")
print(f"p-value: {test_result['p_value']:.6f}")
print(f"Degrees of freedom: {test_result['df']}")
print(f"\nConclusion: {'Reject H0' if test_result['p_value'] < 0.05 else 'Fail to reject H0'} at 5% level")
print(f"Education and performance jointly have {'significant' if test_result['p_value'] < 0.05 else 'no significant'} effect")

## 8. Confidence Intervals

Construct 95% confidence intervals for coefficients:

In [ ]:
# Compute 95% confidence intervals
# Since we used fit_intercept=False, set include_intercept=False
ci = confidence_intervals(result, level=0.95, include_intercept=False)

ci_df = pd.DataFrame({
    'Predictor': coef_names,
    'Estimate': all_coefs,
    'Lower 95%': ci.lower,
    'Upper 95%': ci.upper,
    'Width': ci.upper - ci.lower
})

print("\n95% Confidence Intervals:")
print(ci_df.to_string(index=False))

# Visualize CIs
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(coef_names))

ax.errorbar(all_coefs, y_pos, 
            xerr=[all_coefs - ci.lower, ci.upper - all_coefs],
            fmt='o', markersize=8, capsize=5, capthick=2, linewidth=2)
ax.scatter(true_coefs, y_pos, marker='x', s=100, color='red', 
          label='True value', zorder=5, linewidths=3)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(coef_names)
ax.set_xlabel('Coefficient Value')
ax.set_title('Coefficient Estimates with 95% Confidence Intervals')
ax.legend()
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 9. Model Comparison

Compare full model vs reduced model (without performance):

In [ ]:
# Fit reduced model (no performance)
X_reduced = X[:, :3]  # Intercept, experience, education only

result_reduced = fit_glm(
    X=X_reduced,
    y=y,
    family='gaussian'
)

# Compare models
from scipy.stats import chi2

# Calculate scale (dispersion) for Gaussian GLM
n_params_full = X.shape[1]
n_params_reduced = X_reduced.shape[1]
scale_full = result.deviance_ / (n - n_params_full)
scale_reduced = result_reduced.deviance_ / (n - n_params_reduced)

# Likelihood ratio test
ll_full = -0.5 * (result.deviance_ + n * np.log(2 * np.pi * scale_full))
ll_reduced = -0.5 * (result_reduced.deviance_ + n * np.log(2 * np.pi * scale_reduced))
lr_stat = 2 * (ll_full - ll_reduced)
df_diff = n_params_full - n_params_reduced
p_value_lr = 1 - chi2.cdf(lr_stat, df_diff)

# AIC comparison
aic_full = result.aic_
aic_reduced = result_reduced.aic_

# BIC comparison
bic_full = result.bic_
bic_reduced = result_reduced.bic_

# Calculate R² for both models
y_pred_full = result.predict(X)
y_pred_reduced = result_reduced.predict(X_reduced)
r2_full = 1 - np.sum((y - y_pred_full)**2) / np.sum((y - y.mean())**2)
r2_reduced = 1 - np.sum((y - y_pred_reduced)**2) / np.sum((y - y.mean())**2)

print("\nModel Comparison:")
print("="*60)
print(f"{'Metric':<20} {'Full Model':<15} {'Reduced Model':<15}")
print("="*60)
print(f"{'Deviance':<20} {result.deviance_:<15.2f} {result_reduced.deviance_:<15.2f}")
print(f"{'AIC':<20} {aic_full:<15.2f} {aic_reduced:<15.2f}")
print(f"{'BIC':<20} {bic_full:<15.2f} {bic_reduced:<15.2f}")
print(f"{'R²':<20} {r2_full:.4f}        {r2_reduced:.4f}")
print("="*60)
print(f"\nLikelihood Ratio Test:")
print(f"  LR statistic: {lr_stat:.4f}")
print(f"  df: {df_diff}")
print(f"  p-value: {p_value_lr:.6f}")
print(f"\nConclusion: {'Reject reduced model' if p_value_lr < 0.05 else 'Fail to reject reduced model'}")
print(f" Full model is {'significantly' if p_value_lr < 0.05 else 'not significantly'} better")
print(f" {'Keep' if p_value_lr < 0.05 else 'Remove'} performance predictor")

## 10. Prediction Quality

In [ ]:
# Predictions
y_pred = result.predict(X)

# Metrics
rmse_val = root_mean_squared_error(y, y_pred)
mae_val = mean_absolute_error(y, y_pred)
# Calculate R² manually
r2_val = 1 - np.sum((y - y_pred)**2) / np.sum((y - y.mean())**2)

print("\nPrediction Quality:")
print(f"RMSE: {rmse_val:.2f} ($1000s)")
print(f"MAE: {mae_val:.2f} ($1000s)")
print(f"R²: {r2_val:.4f}")
print(f"\nModel explains {r2_val*100:.1f}% of salary variance")

# Actual vs Predicted plot
plt.figure(figsize=(8, 8))
plt.scatter(y, y_pred, alpha=0.5, edgecolor='k', linewidth=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Salary ($1000s)')
plt.ylabel('Predicted Salary ($1000s)')
plt.title(f'Actual vs Predicted Salary\nR² = {r2_val:.4f}, RMSE = {rmse_val:.2f}')
plt.legend()
plt.grid(alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()

## 11. Prediction with Confidence Intervals

In [ ]:
# Create new observations to predict
new_employees = pd.DataFrame({
    'experience': [2, 5, 10, 15],
    'education': [2, 3, 3, 4],  # BS, MS, MS, PhD
    'performance': [70, 80, 85, 95],
    'description': ['Junior BS', 'Mid-level MS', 'Senior MS', 'Expert PhD']
})

X_new = np.column_stack([
    np.ones(len(new_employees)),
    new_employees['experience'],
    new_employees['education'],
    new_employees['performance']
])

# Point predictions
pred_mean = result.predict(X_new)

# Prediction intervals (approximate)
# SE_pred = sqrt(variance * (1 + X' (X'X)^-1 X))
# Calculate scale (residual variance)
scale = result.deviance_ / (n - X.shape[1])
pred_var = scale * (1 + np.sum(X_new @ np.linalg.inv(X.T @ X) * X_new, axis=1))
pred_se = np.sqrt(pred_var)
pred_lower = pred_mean - 1.96 * pred_se
pred_upper = pred_mean + 1.96 * pred_se

new_employees['predicted_salary'] = pred_mean
new_employees['lower_95'] = pred_lower
new_employees['upper_95'] = pred_upper

print("\nSalary Predictions for New Employees:")
print(new_employees[['description', 'experience', 'education', 'performance', 
                      'predicted_salary', 'lower_95', 'upper_95']].to_string(index=False))

## Best Practices for Linear Regression

1. **Always visualize** your data before modeling
2. **Check diagnostics** after fitting (residual plots)
3. **Interpret carefully** in context of the problem
4. **Report uncertainty** (CIs, SEs) alongside point estimates
5. **Compare models** when choosing predictors
6. **Validate predictions** on held-out data when possible

## Next Steps

- **Heteroscedastic data:** See `01_regression/02_weighted_regression.ipynb`
- **Non-linear relationships:** See `01_regression/03_polynomial_vs_gam.ipynb`
- **Binary outcomes:** See `02_classification/01_logistic_regression.ipynb`

## Resources

- [Aurora-GLM Documentation](https://github.com/Matcraft94/Aurora-GLM)
- [Linear Models with R (Faraway)](https://julianfaraway.github.io/faraway/LMR/)
- [McCullagh & Nelder (1989) - Generalized Linear Models](https://www.routledge.com/Generalized-Linear-Models/McCullagh-Nelder/p/book/9780412317606)